# English → Hindi translation with a text diffusion model (v2, fixed)

A Transformer denoiser learns to remove Gaussian noise from Hindi sentence embeddings, conditioned on
the English sentence through cross-attention. Generation starts from pure noise and denoises the whole
sentence in parallel over 50 steps.

This version fixes the four problems found in `diffusion_LLMs_v1_original.ipynb`:

| # | Problem in v1 | Symptom | Fix |
|---|---|---|---|
| 1 | No positional encoding | attention sees a bag of words; word order lost | sinusoidal positions on both languages |
| 2 | No padding masks | attention and loss dominated by `PAD` tokens | `key_padding_mask` on the source; PAD down-weighted in the loss |
| 3 | Embeddings never trained | `en_embed` / `hi_embed` sat outside the model, so `model.parameters()` skipped them and they stayed random | embeddings are model parameters, plus a rounding loss so they stay decodable |
| 4 | Inference-breaking bug | sampler ran t = 0 → 1 (wrong direction), and decoding referenced an undefined `weights` (NameError) | reverse process t = 1 → 0 with a DDIM-style update; decoding fixed |

All model code lives in `diffusion_mt.py`; this notebook runs it.

In [ ]:
import json, random, time
import torch
from torch.utils.data import DataLoader
import diffusion_mt as dm

random.seed(42); torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

## 1. Data: English–Hindi parallel corpus
`data/train_data1.json` (not committed). The English-Hindi training split has 80,797 sentence pairs; see `data/README.md`.

In [ ]:
tr_s, tr_t, va_s, va_t = dm.load_pairs("data/train_data1.json")
print(f"train pairs: {len(tr_s):,}   validation pairs: {len(va_s):,}")
print(tr_s[123]); print(tr_t[123])

In [ ]:
MAX_LEN = 64
src_vocab = dm.build_vocab(tr_s, min_freq=2)
tgt_vocab = dm.build_vocab(tr_t, min_freq=2)
print(f"English vocab: {len(src_vocab):,}   Hindi vocab: {len(tgt_vocab):,}")

train_dl = DataLoader(dm.ParallelDataset(tr_s, tr_t, src_vocab, tgt_vocab, MAX_LEN), batch_size=64, shuffle=True, drop_last=True)
val_dl = DataLoader(dm.ParallelDataset(va_s, va_t, src_vocab, tgt_vocab, MAX_LEN), batch_size=64)

## 2. Model
English encoder (3 layers) → Transformer denoiser (3 layers of self-attention over the noisy Hindi tokens + cross-attention to the English encoding), with sinusoidal positions and sinusoidal timestep embeddings.

In [ ]:
cfg = dm.Config(len(src_vocab), len(tgt_vocab), max_len=MAX_LEN)
model = dm.TextDiffusionTranslator(cfg).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Fix 3 check: the embedding tables are now trained
assert any(n == "tgt_embed.weight" for n, _ in model.named_parameters())

## 3. Train (8 epochs) and track validation loss

In [ ]:
EPOCHS, LR = 8, 2e-4
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.05)

history = []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for src, tgt in train_dl:
        loss, _ = dm.diffusion_loss(model, src.to(device), tgt.to(device))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); run += loss.item()
    val = dm.evaluate_loss(model, val_dl, device)
    history.append({"epoch": epoch, "train_loss": run / len(train_dl), "val_loss": val})
    print(f"epoch {epoch}/{EPOCHS}  train {run/len(train_dl):.4f}  val {val:.4f}  ({time.time()-t0:.0f}s)")

dm.save_checkpoint("diffusion_mt.pt", model, src_vocab, tgt_vocab)
first, last = history[0]["val_loss"], history[-1]["val_loss"]
print(f"validation loss {first:.2f} -> {last:.2f}")

In [ ]:
import matplotlib.pyplot as plt
ep = [h["epoch"] for h in history]
plt.plot(ep, [h["train_loss"] for h in history], label="train")
plt.plot(ep, [h["val_loss"] for h in history], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Diffusion MT loss"); plt.show()

## 4. Generation check
Translate validation sentences and count failures (a crash or an empty output).

In [ ]:
N = 200
failures, outputs = dm.generation_check(model, va_s[:N], src_vocab, tgt_vocab, device, steps=50)
print(f"{N - failures}/{N} produced output, {failures} generation failures")
for s, ref, out in list(zip(va_s, va_t, outputs))[:8]:
    print(f"\nEN : {s}\nREF: {ref}\nOUT: {out}")

## 5. Optional: BLEU and chrF
Loss shows the model learns; BLEU / chrF++ measure translation quality (`pip install sacrebleu`).

In [ ]:
import sacrebleu
print(sacrebleu.corpus_bleu(outputs, [va_t[:N]]))
print(sacrebleu.corpus_chrf(outputs, [va_t[:N]], word_order=2))

In [ ]:
print(dm.translate(model, ["We are alive.", "Most of the children are born healthy ."], src_vocab, tgt_vocab, device=device))